In [2]:
"""
================================================================
  Pandas vs SQL — Side-by-Side Comparison
  Author  : Vishvi
  Why     : Every Data Scientist must know BOTH.
            This file shows identical operations in SQL and
            Pandas so you can switch between them fluently.
================================================================
"""
!mamba install pandas
import sqlite3, pandas as pd, numpy as np

# ── Setup ────────────────────────────────────────────────
conn = sqlite3.connect(":memory:")

df = pd.DataFrame({
    "name":       ["Vishvi","Aditi","Rohan","Priya","Arjun","Sneha"],
    "department": ["DS","DS","Eng","Eng","Marketing","DS"],
    "salary":     [95000, 85000, 90000, 78000, 70000, 92000],
    "city":       ["Delhi","Mumbai","Bangalore","Delhi","Chennai","Mumbai"],
    "experience": [2, 3, 5, 1, 7, 4]
})
df.to_sql("employees", conn, index=False, if_exists="replace")

def show(title, sql_result, pandas_result):
    print(f"\n{'─'*55}")
    print(f"  📌 {title}")
    print(f"{'─'*55}")
    print("SQL result:")
    print(sql_result.to_string(index=False))
    print("\nPandas result:")
    print(pandas_result.to_string(index=False))

# ── 1. SELECT + WHERE ────────────────────────────────────
sql = pd.read_sql("SELECT name, salary FROM employees WHERE salary > 88000", conn)
pan = df[df["salary"] > 88000][["name","salary"]].reset_index(drop=True)
show("Filter rows (WHERE / boolean indexing)", sql, pan)

# ── 2. ORDER BY ──────────────────────────────────────────
sql = pd.read_sql("SELECT name, salary FROM employees ORDER BY salary DESC", conn)
pan = df[["name","salary"]].sort_values("salary", ascending=False).reset_index(drop=True)
show("Sort rows (ORDER BY / sort_values)", sql, pan)

# ── 3. GROUP BY + AGG ────────────────────────────────────
sql = pd.read_sql("""SELECT department,
                            COUNT(*) AS headcount,
                            ROUND(AVG(salary),0) AS avg_salary,
                            MAX(salary) AS max_salary
                     FROM employees GROUP BY department""", conn)
pan = df.groupby("department").agg(
    headcount=("name","count"),
    avg_salary=("salary", lambda x: round(x.mean(),0)),
    max_salary=("salary","max")
).reset_index()
show("Group + Aggregate (GROUP BY / groupby.agg)", sql, pan)

# ── 4. HAVING ────────────────────────────────────────────
sql = pd.read_sql("""SELECT department, AVG(salary) AS avg_sal
                     FROM employees GROUP BY department
                     HAVING avg_sal > 85000""", conn)
pan = (df.groupby("department")["salary"]
         .mean().reset_index()
         .rename(columns={"salary":"avg_sal"})
         .query("avg_sal > 85000"))
show("Filter groups (HAVING / .query after groupby)", sql, pan)

# ── 5. LIMIT ─────────────────────────────────────────────
sql = pd.read_sql("SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3", conn)
pan = df.nlargest(3, "salary")[["name","salary"]].reset_index(drop=True)
show("Top N rows (LIMIT / nlargest)", sql, pan)

# ── 6. CASE WHEN ─────────────────────────────────────────
sql = pd.read_sql("""SELECT name, salary,
    CASE WHEN salary>=90000 THEN 'Senior'
         WHEN salary>=80000 THEN 'Mid'
         ELSE 'Junior' END AS level
FROM employees""", conn)
pan = df[["name","salary"]].copy()
pan["level"] = pd.cut(df["salary"],
                      bins=[0,79999,89999,999999],
                      labels=["Junior","Mid","Senior"])
show("Conditional column (CASE WHEN / pd.cut)", sql, pan)

# ── 7. NULL handling ─────────────────────────────────────
df_null = df.copy()
df_null.loc[0, "city"] = None
df_null.to_sql("emp_null", conn, index=False, if_exists="replace")

sql = pd.read_sql("SELECT name FROM emp_null WHERE city IS NULL", conn)
pan = df_null[df_null["city"].isna()][["name"]].reset_index(drop=True)
show("Find NULLs (IS NULL / isna)", sql, pan)

# ── 8. String operations ─────────────────────────────────
sql = pd.read_sql("""SELECT name, UPPER(department) AS dept_upper,
                            LENGTH(name) AS name_len
                     FROM employees""", conn)
pan = df[["name","department"]].copy()
pan["dept_upper"] = df["department"].str.upper()
pan["name_len"]   = df["name"].str.len()
show("String ops (UPPER, LENGTH / .str methods)", sql, pan)

# ── 9. Window: Running total ─────────────────────────────
sql = pd.read_sql("""SELECT name, salary,
       SUM(salary) OVER (ORDER BY salary) AS running_total
FROM employees""", conn)
pan = df[["name","salary"]].sort_values("salary").copy()
pan["running_total"] = pan["salary"].cumsum()
show("Running total (SUM OVER / cumsum)", sql, pan)

# ── 10. Window: RANK ─────────────────────────────────────
sql = pd.read_sql("""SELECT name, department, salary,
       RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk
FROM employees""", conn)
pan = df[["name","department","salary"]].copy()
pan["rnk"] = (df.groupby("department")["salary"]
                .rank(method="min", ascending=False)
                .astype(int))
show("Rank within group (RANK OVER PARTITION / groupby rank)", sql, pan)

print("\n" + "="*55)
print("  ✅ Pandas ↔ SQL comparison complete!")
print("  💡 Both tools. One Data Scientist.")
print("="*55)
conn.close()

mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 2.302699999988079 seconds
  Name                          Version                       Build                         Channel                       
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
+ pandas                        3.0.2                         np22py313h9d9dc1e_0           emscripten-forge              
+ python-tzdata                 2026.2                        pyhd8ed1ab_0                  conda-forge                   
- pip                           25.3                          pyh145f28c_0                  conda-forge                   

───────────────────────────────────────────────────────
  📌 Filter rows (WHERE / boolean indexing)
───────────────────────────────────────────────────────
SQL result:
  name 

In [3]:
conn = sqlite3.connect(":memory:")
cur  = conn.cursor()

cur.executescript("""
    CREATE TABLE employees (
        emp_id INT, name TEXT, department TEXT,
        salary REAL, manager_id INT, hire_date TEXT
    );
    CREATE TABLE sales (
        sale_id INT, emp_id INT, region TEXT,
        amount REAL, sale_date TEXT
    );
    CREATE TABLE products (
        product_id INT, name TEXT, category TEXT, price REAL
    );
    CREATE TABLE orders (
        order_id INT, customer_id INT, product_id INT,
        quantity INT, order_date TEXT, status TEXT
    );

    INSERT INTO employees VALUES
        (1,'Ananya','Data Science',95000,5,'2021-03-01'),
        (2,'Bharat','Data Science',85000,5,'2022-06-15'),
        (3,'Chetan','Engineering',90000,6,'2020-11-20'),
        (4,'Divya','Engineering',78000,6,'2023-01-10'),
        (5,'Esha','Data Science',120000,NULL,'2019-08-05'),
        (6,'Farhan','Engineering',115000,NULL,'2018-04-22'),
        (7,'Geeta','Marketing',70000,8,'2022-09-30'),
        (8,'Hari','Marketing',105000,NULL,'2017-12-15'),
        (9,'Isha','Data Science',92000,5,'2021-07-18'),
        (9,'Isha','Data Science',92000,5,'2021-07-18');

    INSERT INTO sales VALUES
        (1,1,'North',15000,'2024-01-10'),
        (2,1,'South',22000,'2024-02-15'),
        (3,2,'North',18000,'2024-01-20'),
        (4,3,'East', 9000,'2024-02-05'),
        (5,2,'South',25000,'2024-03-10'),
        (6,4,'West', 5000,'2024-01-30'),
        (7,1,'East', 30000,'2024-03-05'),
        (8,3,'North',12000,'2024-02-28');

    INSERT INTO products VALUES
        (1,'Laptop','Electronics',75000),
        (2,'Phone','Electronics',25000),
        (3,'Desk','Furniture',12000),
        (4,'Chair','Furniture',8000),
        (5,'Notebook','Stationery',500);

    INSERT INTO orders VALUES
        (101,1,1,2,'2024-01-10','Delivered'),
        (102,2,2,1,'2024-01-15','Delivered'),
        (103,2,3,1,'2024-02-01','Pending'),
        (104,3,1,1,'2024-02-10','Delivered'),
        (105,4,5,5,'2024-02-20','Delivered'),
        (106,5,2,2,'2024-03-01','Cancelled'),
        (107,3,4,3,'2024-03-05','Delivered'),
        (108,1,2,1,'2024-03-10','Delivered');
""")
conn.commit()

def q(level, num, question, query, hint=""):
    print(f"\n{'='*60}")
    print(f"  {level} Q{num}: {question}")
    if hint: print(f"  💡 Hint: {hint}")
    print(f"{'='*60}")
    print(f"SQL:\n{query.strip()}\n")
    df = pd.read_sql_query(query, conn)
    print(df.to_string(index=False))

# ────────────────────────────────────────
print("\n🟢 EASY QUESTIONS")
# ────────────────────────────────────────

q("🟢","1","Find all employees in the Data Science department",
"""SELECT name, salary, hire_date
FROM employees
WHERE department = 'Data Science'
ORDER BY salary DESC;""")

q("🟢","2","Count employees in each department",
"""SELECT department, COUNT(*) AS headcount
FROM employees
GROUP BY department
ORDER BY headcount DESC;""",
"GROUP BY + COUNT")

q("🟢","3","Find employees earning more than the company average",
"""SELECT name, department, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC;""",
"Subquery in WHERE")

q("🟢","4","Get top 3 highest paid employees",
"""SELECT name, department, salary
FROM employees
ORDER BY salary DESC
LIMIT 3;""")

q("🟢","5","Find total sales per employee",
"""SELECT e.name, SUM(s.amount) AS total_sales
FROM employees e
JOIN sales s ON e.emp_id = s.emp_id
GROUP BY e.emp_id
ORDER BY total_sales DESC;""")

# ────────────────────────────────────────
print("\n🟡 MEDIUM QUESTIONS")
# ────────────────────────────────────────

q("🟡","6","Find the second highest salary (classic!)",
"""SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);""",
"Nested MAX with WHERE")

q("🟡","7","Find duplicate rows in employees table",
"""SELECT name, department, salary, COUNT(*) AS duplicates
FROM employees
GROUP BY name, department, salary
HAVING duplicates > 1;""",
"GROUP BY all columns + HAVING COUNT > 1")

q("🟡","8","Get the highest paid employee per department",
"""WITH ranked AS (
    SELECT name, department, salary,
           RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk
    FROM employees
)
SELECT name, department, salary
FROM ranked WHERE rnk = 1;""",
"Window function RANK() OVER PARTITION BY")

q("🟡","9","Find employees with no sales (LEFT JOIN + NULL)",
"""SELECT e.name, e.department
FROM employees e
LEFT JOIN sales s ON e.emp_id = s.emp_id
WHERE s.sale_id IS NULL;""",
"LEFT JOIN — NULL means no match")

q("🟡","10","Monthly sales trend",
"""SELECT SUBSTR(sale_date,1,7) AS month,
       COUNT(*)              AS num_sales,
       SUM(amount)           AS total_revenue
FROM sales
GROUP BY month
ORDER BY month;""")

q("🟡","11","Rank employees by salary within department",
"""SELECT name, department, salary,
       RANK()       OVER (PARTITION BY department ORDER BY salary DESC) AS rank_in_dept,
       DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rank,
       ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS row_num
FROM employees;""",
"RANK vs DENSE_RANK vs ROW_NUMBER — know the difference!")

q("🟡","12","Running total of sales over time",
"""SELECT sale_date, amount,
       SUM(amount) OVER (ORDER BY sale_date) AS running_total
FROM sales
ORDER BY sale_date;""",
"SUM() OVER (ORDER BY ...) = cumulative/running total")

# ────────────────────────────────────────
print("\n🔴 HARD QUESTIONS")
# ────────────────────────────────────────

q("🔴","13","Month-over-month revenue growth %",
"""WITH monthly AS (
    SELECT SUBSTR(sale_date,1,7) AS month,
           SUM(amount)           AS revenue
    FROM sales GROUP BY month
),
with_prev AS (
    SELECT month, revenue,
           LAG(revenue) OVER (ORDER BY month) AS prev_revenue
    FROM monthly
)
SELECT month, revenue, prev_revenue,
       ROUND((revenue - prev_revenue) * 100.0 / prev_revenue, 1) AS pct_growth
FROM with_prev;""",
"CTE + LAG() window function + percentage calculation")

q("🔴","14","Find employees earning more than their manager",
"""SELECT e.name      AS employee,
       e.salary     AS emp_salary,
       m.name       AS manager,
       m.salary     AS mgr_salary
FROM employees e
JOIN employees m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary;""",
"Self JOIN — join a table to itself!")

q("🔴","15","Pivot: total sales by employee per region",
"""SELECT e.name,
       SUM(CASE WHEN s.region='North' THEN s.amount ELSE 0 END) AS North,
       SUM(CASE WHEN s.region='South' THEN s.amount ELSE 0 END) AS South,
       SUM(CASE WHEN s.region='East'  THEN s.amount ELSE 0 END) AS East,
       SUM(CASE WHEN s.region='West'  THEN s.amount ELSE 0 END) AS West
FROM employees e
JOIN sales s ON e.emp_id = s.emp_id
GROUP BY e.emp_id;""",
"PIVOT using CASE WHEN — very common in analytics interviews!")

q("🔴","16","Find customers who ordered in Jan but NOT in Feb (retention)",
"""WITH jan AS (SELECT DISTINCT customer_id FROM orders WHERE SUBSTR(order_date,6,2)='01'),
     feb AS (SELECT DISTINCT customer_id FROM orders WHERE SUBSTR(order_date,6,2)='02')
SELECT j.customer_id AS churned_in_feb
FROM jan j
LEFT JOIN feb f ON j.customer_id = f.customer_id
WHERE f.customer_id IS NULL;""",
"Set difference using LEFT JOIN + NULL — classic retention analysis!")

q("🔴","17","Consecutive days with sales (streak detection)",
"""WITH daily AS (
    SELECT DISTINCT sale_date FROM sales
),
with_grp AS (
    SELECT sale_date,
           DATE(sale_date, '-' || ROW_NUMBER() OVER (ORDER BY sale_date) || ' days') AS grp
    FROM daily
)
SELECT grp, COUNT(*) AS streak_days,
       MIN(sale_date) AS streak_start,
       MAX(sale_date) AS streak_end
FROM with_grp
GROUP BY grp
HAVING streak_days > 1;""",
"Advanced: island-and-gaps problem — senior DS interview question")

print("\n" + "="*60)
print("  ✅ All 17 interview questions solved!")
print("  🟩 Great practice — push this to GitHub!")
print("="*60)

conn.close()


🟢 EASY QUESTIONS

  🟢 Q1: Find all employees in the Data Science department
SQL:
SELECT name, salary, hire_date
FROM employees
WHERE department = 'Data Science'
ORDER BY salary DESC;

  name   salary  hire_date
  Esha 120000.0 2019-08-05
Ananya  95000.0 2021-03-01
  Isha  92000.0 2021-07-18
  Isha  92000.0 2021-07-18
Bharat  85000.0 2022-06-15

  🟢 Q2: Count employees in each department
  💡 Hint: GROUP BY + COUNT
SQL:
SELECT department, COUNT(*) AS headcount
FROM employees
GROUP BY department
ORDER BY headcount DESC;

  department  headcount
Data Science          5
 Engineering          3
   Marketing          2

  🟢 Q3: Find employees earning more than the company average
  💡 Hint: Subquery in WHERE
SQL:
SELECT name, department, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC;

  name   department   salary
  Esha Data Science 120000.0
Farhan  Engineering 115000.0
  Hari    Marketing 105000.0
Ananya Data Science  95000.0

  🟢 Q4: Get top 3

In [4]:
# Connect to in-memory SQLite database
conn = sqlite3.connect(":memory:")
cur  = conn.cursor()

print("=" * 55)
print("  🗄️  SQL for Data Science — Vishvi")
print("=" * 55)

# ──────────────────────────────────────────────────────────
# 🏗️  SETUP: Create sample tables
# ──────────────────────────────────────────────────────────

cur.executescript("""
    CREATE TABLE customers (
        customer_id   INTEGER PRIMARY KEY,
        name          TEXT,
        city          TEXT,
        age           INTEGER,
        signup_date   TEXT
    );

    CREATE TABLE orders (
        order_id      INTEGER PRIMARY KEY,
        customer_id   INTEGER,
        product       TEXT,
        category      TEXT,
        amount        REAL,
        order_date    TEXT,
        status        TEXT
    );

    CREATE TABLE employees (
        emp_id        INTEGER PRIMARY KEY,
        name          TEXT,
        department    TEXT,
        salary        REAL,
        manager_id    INTEGER,
        hire_date     TEXT
    );

    INSERT INTO customers VALUES
        (1,  'Vishvi',   'Delhi',     23, '2023-01-15'),
        (2,  'Aditi',    'Mumbai',    28, '2022-06-20'),
        (3,  'Rohan',    'Bangalore', 32, '2021-03-10'),
        (4,  'Priya',    'Delhi',     25, '2023-08-05'),
        (5,  'Arjun',    'Chennai',   35, '2020-11-30'),
        (6,  'Sneha',    'Mumbai',    27, '2022-09-15'),
        (7,  'Vikram',   'Delhi',     30, '2021-07-22'),
        (8,  'Meera',    'Pune',      29, '2023-02-18');

    INSERT INTO orders VALUES
        (101, 1, 'Laptop',     'Electronics', 75000, '2024-01-10', 'Delivered'),
        (102, 1, 'Phone',      'Electronics', 25000, '2024-02-15', 'Delivered'),
        (103, 2, 'Headphones', 'Electronics',  3500, '2024-01-20', 'Delivered'),
        (104, 3, 'Desk',       'Furniture',   12000, '2024-03-05', 'Pending'),
        (105, 3, 'Chair',      'Furniture',    8000, '2024-03-06', 'Delivered'),
        (106, 4, 'Notebook',   'Stationery',    500, '2024-02-28', 'Delivered'),
        (107, 5, 'Monitor',    'Electronics', 18000, '2024-01-15', 'Cancelled'),
        (108, 6, 'Keyboard',   'Electronics',  2000, '2024-03-10', 'Delivered'),
        (109, 7, 'Laptop',     'Electronics', 80000, '2024-02-05', 'Delivered'),
        (110, 2, 'Mouse',      'Electronics',  1500, '2024-03-01', 'Delivered'),
        (111, 8, 'Lamp',       'Furniture',    2500, '2024-01-30', 'Pending'),
        (112, 5, 'Phone',      'Electronics', 30000, '2024-03-20', 'Delivered');

    INSERT INTO employees VALUES
        (1, 'Ananya',  'Data Science', 95000,  5, '2021-03-01'),
        (2, 'Bharat',  'Data Science', 85000,  5, '2022-06-15'),
        (3, 'Chetan',  'Engineering',  90000,  6, '2020-11-20'),
        (4, 'Divya',   'Engineering',  78000,  6, '2023-01-10'),
        (5, 'Esha',    'Data Science', 120000, NULL,'2019-08-05'),
        (6, 'Farhan',  'Engineering',  115000, NULL,'2018-04-22'),
        (7, 'Geeta',   'Marketing',    70000,  8, '2022-09-30'),
        (8, 'Hari',    'Marketing',    105000, NULL,'2017-12-15'),
        (9, 'Isha',    'Data Science', 92000,  5, '2021-07-18');
""")
conn.commit()

def run(title, query):
    """Run a SQL query and display results as a DataFrame."""
    print(f"\n{'─'*55}")
    print(f"  📌 {title}")
    print(f"{'─'*55}")
    print(f"SQL:\n{query.strip()}")
    print()
    df = pd.read_sql_query(query, conn)
    print(df.to_string(index=False))
    return df


# ──────────────────────────────────────────────────────────
# 1️⃣  BASIC SELECT
# ──────────────────────────────────────────────────────────

run("All customers", "SELECT * FROM customers;")

run("Select specific columns",
"""SELECT name, city, age
FROM customers
ORDER BY age DESC;""")

run("Filter with WHERE",
"""SELECT name, city
FROM customers
WHERE city = 'Delhi';""")

run("Multiple conditions (AND / OR)",
"""SELECT name, city, age
FROM customers
WHERE city = 'Mumbai' OR age < 26;""")


# ──────────────────────────────────────────────────────────
# 2️⃣  AGGREGATIONS & GROUP BY
# ──────────────────────────────────────────────────────────

run("Total revenue",
"""SELECT SUM(amount) AS total_revenue
FROM orders
WHERE status = 'Delivered';""")

run("Orders per customer",
"""SELECT customer_id,
       COUNT(*)        AS total_orders,
       SUM(amount)     AS total_spent,
       AVG(amount)     AS avg_order_value,
       MAX(amount)     AS biggest_order
FROM orders
GROUP BY customer_id
ORDER BY total_spent DESC;""")

run("Revenue by category",
"""SELECT category,
       COUNT(*)    AS num_orders,
       SUM(amount) AS revenue
FROM orders
GROUP BY category
ORDER BY revenue DESC;""")

run("HAVING — customers who spent over 20,000",
"""SELECT customer_id, SUM(amount) AS total_spent
FROM orders
GROUP BY customer_id
HAVING total_spent > 20000
ORDER BY total_spent DESC;""")


# ──────────────────────────────────────────────────────────
# 3️⃣  JOINS
# ──────────────────────────────────────────────────────────

run("INNER JOIN — orders with customer names",
"""SELECT c.name, o.product, o.amount, o.status
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.amount DESC;""")

run("LEFT JOIN — all customers, even with no orders",
"""SELECT c.name, c.city, COUNT(o.order_id) AS num_orders
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id
ORDER BY num_orders DESC;""")

run("JOIN + GROUP BY — top spending customers",
"""SELECT c.name, c.city, SUM(o.amount) AS total_spent
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'Delivered'
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 5;""")


# ──────────────────────────────────────────────────────────
# 4️⃣  SUBQUERIES
# ──────────────────────────────────────────────────────────

run("Subquery — customers who placed orders",
"""SELECT name, city
FROM customers
WHERE customer_id IN (
    SELECT DISTINCT customer_id FROM orders
);""")

run("Subquery — orders above average amount",
"""SELECT product, amount
FROM orders
WHERE amount > (SELECT AVG(amount) FROM orders)
ORDER BY amount DESC;""")

run("Correlated subquery — each customer vs avg",
"""SELECT c.name,
       o.product,
       o.amount,
       ROUND((SELECT AVG(amount) FROM orders), 2) AS avg_all_orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.amount > (SELECT AVG(amount) FROM orders);""")


# ──────────────────────────────────────────────────────────
# 5️⃣  CTEs (Common Table Expressions)
# ──────────────────────────────────────────────────────────

run("CTE — high-value customers (spent > 50,000)",
"""WITH high_value AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM orders
    WHERE status = 'Delivered'
    GROUP BY customer_id
    HAVING total_spent > 50000
)
SELECT c.name, c.city, hv.total_spent
FROM customers c
JOIN high_value hv ON c.customer_id = hv.customer_id
ORDER BY hv.total_spent DESC;""")

run("CTE — monthly revenue trend",
"""WITH monthly AS (
    SELECT SUBSTR(order_date, 1, 7) AS month,
           SUM(amount)              AS revenue,
           COUNT(*)                 AS num_orders
    FROM orders
    WHERE status = 'Delivered'
    GROUP BY month
)
SELECT month, revenue, num_orders
FROM monthly
ORDER BY month;""")


# ──────────────────────────────────────────────────────────
# 6️⃣  WINDOW FUNCTIONS
# ──────────────────────────────────────────────────────────

run("ROW_NUMBER — rank orders by amount",
"""SELECT product, amount,
       ROW_NUMBER() OVER (ORDER BY amount DESC) AS rank
FROM orders
WHERE status = 'Delivered';""")

run("RANK — salary rank within department",
"""SELECT name, department, salary,
       RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank
FROM employees;""")

run("Running total — cumulative revenue over time",
"""SELECT order_date, amount,
       SUM(amount) OVER (ORDER BY order_date) AS running_total
FROM orders
WHERE status = 'Delivered'
ORDER BY order_date;""")

run("LAG — compare each order to previous order amount",
"""SELECT product, order_date, amount,
       LAG(amount) OVER (ORDER BY order_date)              AS prev_amount,
       amount - LAG(amount) OVER (ORDER BY order_date)     AS change
FROM orders
WHERE status = 'Delivered'
ORDER BY order_date;""")

run("LEAD — look ahead to next order",
"""SELECT product, order_date, amount,
       LEAD(amount) OVER (ORDER BY order_date) AS next_order_amount
FROM orders
WHERE status = 'Delivered'
ORDER BY order_date;""")


# ──────────────────────────────────────────────────────────
# 7️⃣  CASE WHEN
# ──────────────────────────────────────────────────────────

run("CASE WHEN — segment customers by spend",
"""SELECT c.name,
       SUM(o.amount) AS total_spent,
       CASE
           WHEN SUM(o.amount) >= 50000 THEN 'Premium'
           WHEN SUM(o.amount) >= 10000 THEN 'Regular'
           ELSE 'Occasional'
       END AS customer_segment
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC;""")

run("CASE WHEN — salary band classification",
"""SELECT name, department, salary,
       CASE
           WHEN salary >= 100000 THEN 'Senior'
           WHEN salary >= 85000  THEN 'Mid-level'
           ELSE 'Junior'
       END AS level
FROM employees
ORDER BY salary DESC;""")


# ──────────────────────────────────────────────────────────
# 8️⃣  NULL HANDLING
# ──────────────────────────────────────────────────────────

run("IS NULL — employees with no manager (leaders)",
"""SELECT name, department, salary
FROM employees
WHERE manager_id IS NULL;""")

run("COALESCE — replace NULL with a default",
"""SELECT name,
       COALESCE(CAST(manager_id AS TEXT), 'No Manager') AS manager
FROM employees;""")


# ──────────────────────────────────────────────────────────
# 9️⃣  STRING & DATE FUNCTIONS
# ──────────────────────────────────────────────────────────

run("String functions — UPPER, LENGTH, SUBSTR",
"""SELECT name,
       UPPER(city)         AS city_upper,
       LENGTH(name)        AS name_length,
       SUBSTR(name, 1, 3)  AS name_prefix
FROM customers;""")

run("Date functions — extract year from signup",
"""SELECT name,
       signup_date,
       SUBSTR(signup_date, 1, 4) AS signup_year,
       SUBSTR(signup_date, 6, 2) AS signup_month
FROM customers
ORDER BY signup_date;""")


# ──────────────────────────────────────────────────────────
# 🔟  INTERVIEW PATTERNS
# ──────────────────────────────────────────────────────────

run("Interview: Find duplicate products in orders",
"""SELECT product, COUNT(*) AS times_ordered
FROM orders
GROUP BY product
HAVING times_ordered > 1
ORDER BY times_ordered DESC;""")

run("Interview: Top N per group (top earner per dept)",
"""WITH ranked AS (
    SELECT name, department, salary,
           RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk
    FROM employees
)
SELECT name, department, salary
FROM ranked
WHERE rnk = 1;""")

run("Interview: Month-over-month revenue change",
"""WITH monthly AS (
    SELECT SUBSTR(order_date, 1, 7) AS month,
           SUM(amount)              AS revenue
    FROM orders
    WHERE status = 'Delivered'
    GROUP BY month
),
with_prev AS (
    SELECT month, revenue,
           LAG(revenue) OVER (ORDER BY month) AS prev_revenue
    FROM monthly
)
SELECT month,
       revenue,
       prev_revenue,
       ROUND(((revenue - prev_revenue) * 100.0 / prev_revenue), 1) AS pct_change
FROM with_prev;""")

run("Interview: Customers with no orders (churn candidates)",
"""SELECT c.name, c.city, c.signup_date
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;""")

run("Interview: Second highest salary",
"""SELECT MAX(salary) AS second_highest_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);""")

print("\n" + "="*55)
print("  ✅ All SQL queries executed successfully!")
print("  🟩 Push this file to GitHub for today's green square!")
print("="*55)

conn.close()

  🗄️  SQL for Data Science — Vishvi

───────────────────────────────────────────────────────
  📌 All customers
───────────────────────────────────────────────────────
SQL:
SELECT * FROM customers;

 customer_id   name      city  age signup_date
           1 Vishvi     Delhi   23  2023-01-15
           2  Aditi    Mumbai   28  2022-06-20
           3  Rohan Bangalore   32  2021-03-10
           4  Priya     Delhi   25  2023-08-05
           5  Arjun   Chennai   35  2020-11-30
           6  Sneha    Mumbai   27  2022-09-15
           7 Vikram     Delhi   30  2021-07-22
           8  Meera      Pune   29  2023-02-18

───────────────────────────────────────────────────────
  📌 Select specific columns
───────────────────────────────────────────────────────
SQL:
SELECT name, city, age
FROM customers
ORDER BY age DESC;

  name      city  age
 Arjun   Chennai   35
 Rohan Bangalore   32
Vikram     Delhi   30
 Meera      Pune   29
 Aditi    Mumbai   28
 Sneha    Mumbai   27
 Priya     Delhi   25